# သင်ခန်းစာ ၁၃ - Agent သတိရမှု


## စတင်ပြင်ဆင်ခြင်း

ဒီ notebook က **Microsoft Agent Framework** (MAF) ကို အသုံးပြုကာ **ကိုယ်တိုင်မှတ်ဉာဏ်** ပါရှိတဲ့ ခရီးသွားဘွတ်ကင်အေးဂျင့်တစ်ခု တည်ဆောက်နည်းကို ပြသပေးပါသည်။

အမျိုးမျိုးသော အေးဂျင့်မှတ်ဉာဏ်အမျိုးအစားများ — အလုပ်လုပ်မှု၊ တိုတို၊ နှင့် ရောဂါရှည် — တို့က အေးဂျင့်က ပြောဆိုမှုများအတွင်း သတင်းအချက်အလက်ကို ထိန်းသိမ်းပြီး အသုံးပြုနည်းအား ဘယ်လို ထိခိုက်သက်ရောက်မှုရှိစေသည်ကို သင်ယူပါလိမ့်မည်။

**လိုအပ်ချက်များ -**
- chat model တစ်ခု တပ်ဆင်ပြီး 있는 Microsoft Foundry project (ဥပမာ `gpt-5-mini`) တစ်ခု။
- Azure CLI ဖြင့် ဝင်ရောက်ထားပြီး — သင့် terminal မှာ `az login` ကို လည်ပတ်ပါ။
- `AZURE_AI_PROJECT_ENDPOINT` — မိုက်ခရိုဆော့ ဖောင်ထရီ ပရောဂျက် endpoint သင့်။
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — သင့်တပ်ဆင်ထားသော model အမည်။


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## အေးဂျင့်မှတ်ဉာဏ် အမျိုးအစားများ

AI အေးဂျင့်များသည် အမျိုးမျိုးသောမှတ်ဉာဏ်များကို အသုံးချနိုင်ပြီး၊ တစ်ခုချင်းစီမှာ မတူညီသည့်ရည်ရွယ်ချက်ရှိပါသည်။

### အလုပ်လုပ်မှတ်ဉာဏ်
စကားပြောရှုးလမ်းကြောင်းကိုယ်တိုင် — တစ်ကြိမ်ခါဆွေးနွေးချက်အတွင်း အကြောင်းအရာများလဲလှယ်ခြင်း။ အေးဂျင့်သည် တစ်ဦးချင်းစီ၏ စကားပြောရှုးလမ်းကြောင်းများထဲမှ အစဉ်လိုက်အချက်အလက်များကို ပြန်ကြည့်၍ ညီညွတ်မှုကို ထိန်းသိမ်းနိုင်သည်။ MAF တွင် ဒီအခန်းကဏ္ဍကို **`agent.create_session()`** မှတဆင့်ဖန်တီးပြီး `AgentSession` ကို ပြန်လည်ရယူသည်။

### အတိုချုပ်မှတ်ဉာဏ်
တာဝန်တစ်ခု သို့မဟုတ် ဆွေးနွေးချက်တစ်ခုချင်းစီသည် ကျန်ရှိသော်လည်း အစဉ်အမြဲ မသိမ်းဆည်းထားသော အချက်အလက်များ။ ဥပမာ၊ အေးဂျင့်သည် မျိုးစုံပြန်လည်ဆွေးနွေးချက်တွင် ခြေလှမ်းအစဉ်များကို စုဆောင်းကာ နောက်ဆုံး အစီအစဉ်ကို ထုတ်ပေးနိုင်သည်။

### ရေရှည်မှတ်ဉာဏ်
အစဉ်အမြဲရှိနေသော နှစ်ဦးအကြား ထိုက်တန်မှုများနှင့်အတူ မှတ်ဉာဏ်များ။ ပြန်လာသော အသုံးပြုသူသည် သူတို့၏ အစားအသောက် ကန့်သတ်ချက်များ သို့မဟုတ် ခရီးသွားစနစ်များကို ထပ်မံပြောဆိုရန် မလိုအပ်ပါ။ ရေရှည်မှတ်ဉာဏ်ကို အများအားဖြင့် ပြင်ပသိုလှောင်မှုဖြစ်သော ဒေတာဘေ့(စ်), ဖိုင် သို့ vector index တို့ဖြင့် ထောက်ပံ့ပြီး အေးဂျင့်ဆီသို့ ကိရိယာများမှတဆင့် ပြသသည်။


## အလုပ်လုပ်မှတ်ဉာဏ် နှင့် အစည်းအဝေးများ

မှတ်ဉာဏ်ရဲ့အရိုးရိုးဆုံးပုံစံက စကားပြော အစည်းအဝေးဖြစ်ပါတယ်။ တူညီတဲ့အစည်းအဝေး ( `agent.create_session()` ဖြင့် ဖန်တီးထားသည်) ကို ဆက်တိုက် `agent.run()` ခေါ်ဆိုမှုများသို့ ပေးပို့လျှင်၊ အဲဒီယောက်ျားက ဤစကားပြောမှတ်တမ်းအပြည့်အစုံကို မြင်နေရုံမက အရင်ကအသေးစိတ်တွေကို ပြန်လည်တွေးမိနိုင်ပါတယ်။

ခရီးသွားငွေရေးအတွက် ဉီးစီးသူတစ်ဦး ဖန်တီးပြီး အလုပ်လုပ်မှတ်ဉာဏ်ကို ဖော်ပြကြစို့။


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

အေးဂျင့်သည် ဘတ်ဂျက်ကိုမှန်ကန်စွာ သတိရနိုင်ခဲ့သည်မှာ နှစ်ခုစလုံးပေးပို့ချက်များသည် တူညီသော session တစ်ခုကိုမျှဝေသောကြောင့်ဖြစ်သည်။ ဒါဟာ **အလုပ်လုပ်နေသောမှတ်ဉာဏ်** ဖြစ်ပြီး — session ၏အသက်တာအတွက်သာ ရှိနေသည်။

### သစ်သတင်းစဉ်အသစ်နဲ့ဘာဖြစ်မလဲ?

ကျွန်ုပ်တို့သည် **သစ်** session အသစ်တစ်ခုဖန်တီးပါက အေးဂျင့်တွင် ယခင်စကားပြောနောက်ခံကို မမှတ်မိနိုင်ပါ။


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## အချိန်ရှည်မှတ်ဉာဏ်ပုံစံ

အသုံးပြုသူနှစ်သက်ချက်များကို **အစောင့်ကြည့်စဉ်တစ်လျှောက်လုံး** မှတ်မိထားရန်၊ စကားပြောခြင်းတန်းထဲမှအပြင်တွင် အသက်ရှင်သော အစဉ်တစိုက်မှတ်တမ်းဆိုင်ရာနေရာတစ်ခု လိုအပ်သည်။ ကိုယ်စားလှယ်သည် ဤနေရာကို **ကိရိယာများ** ဖြင့် အသုံးပြုသည် — သိမ်းဆည်းရယူစရာ အချက်အလက်များကို ခေါ်ယူနိုင်သော စနစ်များဖြစ်သည်။

အောက်တွင် သိုလှောင်မှုကို ရိုးရှင်းသမျှ ပုံမှန်မှတ်ဉာဏ်ပုံစံတစ်ခုအား (ထုတ်လုပ်မှုတွင် ဒေတာဘေ့စ် သို့မဟုတ် vector အညွှန်းနှင့်ထောက်ပံ့မည်) ကိုတည်ဆောက်ပြီး ကိုယ်စားလှယ်အသုံးပြုနိုင်ရန် ကိရိယာများအဖြစ် ထုတ်ဖော်ပြသထားသည်။

### နောက်ခံဖွဲ့စည်းပုံ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### အခန်းကဏ္ဍ ၁ — ပထမဆုံးအသုံးပြုသူသည် နှစ်ပတ်လည်ခရီးစဉ်ကို မှာယူသည်

Sarah သည် ပထမဆုံး ဧည့်ထွက်လာသည်။ ကိုယ်စားလှယ်သည်သူမ၏ရွေးချယ်စရာများကို ကိရိယာများမှတဆင့် ထိန်းသိမ်းပြီး ဟိုတယ်များကို အကြံပြုရန် အသုံးပြုသင့်သည်။


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### အခြေအနေ ၂ — ဆာရာ ပတ်လည်သတ္တိတစ်ပတ်လောက် ပြန်လာသည်

ဆာရာသည် **အလွန်သစ်သော စကားဝိုင်း** တစ်ခုပြုလုပ်သည် (အစည်းအဝေးအသစ်ကို စားသုံးသည့်အတိုင်းလှုပ်ရှားခြင်းဖြစ်သည်)။ အလုပ်လုပ်သော မှတ်ဉာဏ်မှာ ဗလာပြီးသားဖြစ်ပေမယ့် ရှည်လျားသိမ်းဆည်းထားသောနှစ်သက်မှုဆိုင်ရာ သိမ်းဆည်းမှုမှာ သူမ၏ အချက်အလက်ကို ထားရှိနေသည်။ ဂျင့်တင်သည် အဲဒီအချက်အလက်ကို ထုတ်ယူပြီး ကိုယ်ပိုင်သုံးသပ်စဉ်းစားမှုများ ပေးစွမ်းရန် သင့်တော်သည်။


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## အကျဉ်းချုပ်

ဒီသင်ခန်းစာတွင် သင်သည် agent မှတ်ဉာဏ်အမျိုးအစားသုံးမျိုးနှင့် Microsoft Agent Framework ဖြင့် အဲဒီများကို ဘယ်လိုသုံးစွဲရမည်ကို လေ့လာခဲ့သည်။

| မှတ်ဉာဏ်အမျိုးအစား | MAF စနစ် | အသက်ကာလ |
|---|---|---|
| **အလုပ်လုပ်ဆောင်မှု** | `agent.create_session()` | စကားပြောဆိုမှုတစ်ခုပြီးဆုံးသည်အထိ |
| **တိုတောင်းသော** | တစ်ခုတည်းသောအလုပ်တစ်ခု / အစည်းအဝေးအတွင်းစုဆောင်းမှု | တစ်ခုတည်းသောအလုပ် / အစည်းအဝေး |
| **ရှည်လျားသော** | `@tool` ဆောင်ရွက်ချက်များမှတဆင့်အသုံးပြုသော ပြင်ပသိုလှောင်မှု | အစည်းအဝေးများအတွင်းကျော်လွန်နိုင်သည် |

### အဓိကယူဆချက်များ
1. **`agent.create_session()`** သည် အလုပ်လုပ်ဆောင်မှုမှတ်ဉာဏ်ကို ပေးသည် — agent သည် session တစ်ခုအတွင်း စကားပြောဆိုမှုမှတ်တမ်းအပြည့်အစုံကို ကြည့်ရှုနိုင်သည်။
2. **အစည်းအဝေးအသစ်များတွင် စဉ်းစားမှုမရှိတော့ပါ** — ရှည်လျားသောမှတ်ဉာဏ်မရှိလျှင် agent သည် မကြာသေးမီက စကားများကို မမှတ်မိနိုင်ပါ။
3. **`@tool` functions** သည် ကြားခံကွန်နက်ရှင်းဖြစ်စေသည် — အသေးစိတ်သိမ်းဆည်းမှုမှ agent သည် သတင်းအချက်အလက်များကို သိမ်းဆည်းပြီး ဆွဲထုတ်နိုင်သည်။
4. **ပုဂ္ဂိုလ်ရေးစိတ်ကြိုက်မှုက အချိန်ကုန်လျော့လာသည်နှင့်တပြေးညီ တိုးတတ်လာသည်** — စိတ်ကြိုက်မှုများပိုမိုသိမ်းဆည်းသွားသည်နှင့်အမွေခံအကြံပြုချက်များ ပိုကောင်းလာသည်။

### အကောင်အထည်ဖော်နိုင်မှုများ
- **ဖောက်သည်ဝန်ဆောင်မှု**: ဖောက်သည်၏သမိုင်းနှင့် စိတ်ကြိုက်မှုများမှတ်တမ်းထားရေး
- **ပုဂ္ဂိုလ်ရေးအကူပြုသူများ**: ရက်ပေါင်းများ သို့မဟုတ် အပတ်ပေါင်းများအတွင်း စဉ်းစားမှုကို သိမ်းဆည်းထားနိုင်ရန်
- **ကျန်းမာရေး**: လူနာအချက်အလက်များနှင့် စိတ်ကြိုက်မှု တတ်မြောက်စွာစောင့်ကြည့်ရေး
- **အီလက်ထရွန်နစ်စျေးကွက်**: သမိုင်းအရ စိတ်ကြိုက်စျေးဝယ်ခြင်း

### နောက်တစ်ဆင့်များ
- မှတ်ဉာဏ်အတွင်းရှိ ဒစ်ရှင်နယ်ရီကို ဒေတာဘေစ်စ် သို့မဟုတ် ဝက်ဖ်တာစတိုး (ဥပမာ Azure AI Search) ဖြင့် အစားထိုးပါ
- အချိန်အားပေးသတင်းအချက်အလက်များအတွက် မှတ်ဉာဏ် သက်တံ့မှီမှု ထည့်သွင်းပါ
- မှတ်ဉာဏ်မျှဝေမှုပါဝင်သော များစုံ Agent စနစ်များ တည်ဆောက်ပါ
- Cognee notebook ကို သုတေသနပြုပြီး သိပ္ပံအခြေခံ မွတ်ဉာဏ်စနစ်ကို စူးစမ်းပါ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
